# Extract data + Spectrogram generation

We first get the full database and then extract the data, separating training and test data. From this we extract two pandas dataframes.

In [ ]:
import audb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import torchvision.models as models
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torch
import torchinfo

from PIL import Image
import json
import urllib.request

import torch
import torchaudio
import torchaudio.transforms as T

# Load the emodb database
db = audb.load("emodb")

Get:   emodb v2.0.0
Cache: /home/pierre/audb/emodb/2.0.0/d3b62a9b


In [16]:
# get the data, no ambiguous audios, separate train and test splits

train_data: pd.DataFrame = db["emotion.categories.train.gold_standard"].get().copy()
test_data: pd.DataFrame = db["emotion.categories.test.gold_standard"].get().copy()

From the audios given, we make them 3s long by padding or truncating them. Then, we calculate their spectrogram and add it to the dataframe to make it usable.

In [ ]:
def truncate_or_pad(x, target_len):
    current_len = x.shape[1]

    if isinstance(x, torch.Tensor):
        if current_len > target_len:
            return x[:, :target_len]
        else:
            pad_size = target_len - current_len
            pad_shape = (1, pad_size)
            padding = torch.zeros(pad_shape, dtype=x.dtype, device=x.device)
            return torch.cat([x, padding], dim=1)


def compute_spectrogram(file_path):
    waveform, sample_rate = torchaudio.load(file_path)
    waveform = truncate_or_pad(waveform, sample_rate * 3)

    # 25ms windows every 10ms
    window_length = int(sample_rate * 0.025)  # 25ms frames
    hop_length = int(sample_rate * 0.010)  # 10ms hop

    mel_transform = T.MelSpectrogram(
        sample_rate=sample_rate,
        win_length=window_length,
        hop_length=hop_length,
        n_mels=64,
    )

    amp_to_db = T.AmplitudeToDB(stype="power", top_db=80)

    spec_db = amp_to_db(mel_transform(waveform)).squeeze(0)

    return spec_db


def add_spectrogram_col(datafram_data):
    spectrogram_list = []
    for file_path in datafram_data.index:
        spectrogram_list.append(compute_spectrogram(file_path))

    datafram_data.insert(1, "spectrogram", spectrogram_list)

Finally, we can verify the number of audios for each of the splits, and show some spectrograms we just computed.

In [31]:
print("Number of training audios: ", len(train_data))
print("Number of testing audios:  ", len(test_data))

# plt.imshow(
#     compute_spectrogram(train_data.index[1]).numpy(),
#     origin="lower",
#     aspect=301 / 64,
#     cmap="gray",
# )
# plt.axis("off")
# plt.show()
# plt.imshow(
#     compute_spectrogram(test_data.index[1]).numpy(),
#     origin="lower",
#     aspect=301 / 64,
#     cmap="gray",
# )
# plt.axis("off")
# plt.show()

Number of training audios:  304
Number of testing audios:   231


We create the function that will save the spectrograms as images so that it can be the input of the ResNet18 model

In [ ]:
# Function to save a 224*224*3 image of a spectrogram, with the riht filename, for a given spectrogram


def make_spectrogram_image(input_path, output_path):
    # Compute spectrogram data
    spectrogram = compute_spectrogram(input_path)
    figure = plt.figure(figsize=(2.24, 2.24), dpi=100)
    axes = figure.add_axes([0, 0, 1, 1])
    axes.imshow(
        spectrogram.numpy(),
        origin="lower",
        aspect="auto",
        cmap="gray",
    )
    axes.axis("off")
    figure.savefig(
        output_path,
        dpi=100,
        pad_inches=0,
    )
    plt.close(figure)

We make all the spectrograms png files

In [ ]:
if not os.path.exists("test_png"):
    os.makedirs("test_png")

if not os.path.exists("train_png"):
    os.makedirs("train_png")

for file_path in test_data.index:
    emotion = test_data.loc[file_path, "emotion"]
    if not os.path.exists(f"test_png/{emotion}"):
        os.makedirs(f"test_png/{emotion}")
    output_path = f"test_png/{emotion}/{file_path.split('/')[-1].split('.')[0]}.png"
    make_spectrogram_image(file_path, output_path)

for file_path in train_data.index:
    emotion = train_data.loc[file_path, "emotion"]
    if not os.path.exists(f"train_png/{emotion}"):
        os.makedirs(f"train_png/{emotion}")
    output_path = f"train_png/{emotion}/{file_path.split('/')[-1].split('.')[0]}.png"
    make_spectrogram_image(file_path, output_path)

## ResNet18 model configuration

In [20]:
import torch
import torch.nn as nn

# Import Model
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)

In [38]:
torchinfo.summary(model, input_size=(1, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 7]                    --
├─Conv2d: 1-1                            [1, 64, 112, 112]         (9,408)
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         (128)
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 64, 56, 56]           --
│    └─BasicBlock: 2-1                   [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           (36,864)
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           (128)
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           (36,864)
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           (128)
│    │    └─ReLU: 3-6                    [1, 64, 56, 56]   

### Modify the model to fit our needs

With frozen backbone, we will fit a new fully connected head

In [ ]:
# First, we freeze all the layers of the model, except for the last fully connected layer
for param in model.parameters():
    param.requires_grad = False

# then we create a new fc layer with only the number of classes we want (we have 7 emotions)
model.fc = nn.Linear(model.fc.in_features, 7)

### Prepare the datasets for training

In [26]:
# prepare the transforms
# as we expect the pngs to already be 224x224, we only need to convert them to tensors
# TODO keep normalizing them ? YES because it is not a from scratch model, and ResNet18 was trained on normalized images

transform_spectrogram = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_dataset = ImageFolder("train_png", transform=transform_spectrogram)
test_dataset = ImageFolder("test_png", transform=transform_spectrogram)

### Prepare the training settings

In [29]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=2
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)

# Put the model on GPU if available

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(device)
print(train_dataset.class_to_idx)

cpu
{'anger': 0, 'boredom': 1, 'disgust': 2, 'fear': 3, 'happiness': 4, 'neutral': 5, 'sadness': 6}


In [ ]:
# Select the criterion and optimizer
criterion = (
    torch.nn.CrossEntropyLoss()
)  # according to the internet it is standard for multi-class
optimizer = torch.optim.Adam(
    model.fc.parameters(), lr=0.001
)  # According to the internet, Adam is easier to tune than SGD

# TODO modify learning rate to have a lower lr at the end of the training ?

### Finally, training the model

In [ ]:
# The number of epochs can be adjuster here or we can launch the cell multiple times to continue the training
num_epochs = 5

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    # Training
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        # Put data on the right device
        images = images.to(device)
        labels = labels.to(device)

        # Reset the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass with optimising
        loss.backward()
        optimizer.step()

        # Compute the stats
        train_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    # Evaluation
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            test_correct += (predictions == labels).sum().item()
            test_total += labels.size(0)

    # Normalize the losses and accuracies
    train_loss /= train_total
    test_loss /= test_total
    train_accuracy = train_correct / train_total
    test_accuracy = test_correct / test_total

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train loss: {train_loss:.4f} | "
        f"Train accuracy: {train_accuracy:.2%} | "
        f"Test loss: {test_loss:.4f} | "
        f"Test accuracy: {test_accuracy:.2%}"
    )

print("\nTraining complete!")


Epoch 1/5
Epoch 1/5 | Train loss: 0.8307 | Train accuracy: 77.30% | Test loss: 1.2423 | Test accuracy: 47.19%

Epoch 2/5
Epoch 2/5 | Train loss: 0.7777 | Train accuracy: 79.93% | Test loss: 1.2222 | Test accuracy: 52.81%

Epoch 3/5
Epoch 3/5 | Train loss: 0.7468 | Train accuracy: 82.57% | Test loss: 1.2004 | Test accuracy: 54.11%

Epoch 4/5
Epoch 4/5 | Train loss: 0.6932 | Train accuracy: 84.21% | Test loss: 1.1979 | Test accuracy: 52.38%

Epoch 5/5
Epoch 5/5 | Train loss: 0.6948 | Train accuracy: 86.18% | Test loss: 1.1860 | Test accuracy: 51.95%

Training complete!


In [36]:
# When satisfied with a training, save the model
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_to_idx": train_dataset.class_to_idx,
    },
    "resnet18_emotion.pth",
)

# Also save only the fc layer weights:

torch.save(model.fc.state_dict(), "resnet18_emotion_fc.pth")